# 循环神经网络与自回归语言模型 (RNN-LM) 企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 NLP / 语言模型算法岗面试手撕  
> **核心涵盖**：Vanilla RNNCell 纯微积分求导、TimeRNN 时序展开、截断反向传播 (Truncated BPTT)、梯度裁剪、GRU 门控单元、LSTM 细胞高速公路、自回归因果错位预测、PPL 困惑度评估与温度采样  
> **设计准则**：纯 NumPy 底层白盒手推，彻底解构记忆传递、梯度消失/爆炸与时序展开计算图。

---
### 核心模块速览
1. **模块一**：单步 Vanilla RNNCell 纯 NumPy 白盒前向与反向传播求导
2. **模块二**：TimeRNN 时序展开与 BPTT 全量逆流梯度手撕
3. **模块三**：梯度裁剪 (Gradient Clipping) 算法手撕 (L2 范数安全制动)
4. **模块四**：Truncated BPTT 截断反向传播手撕 (隐状态跨 Batch 传递与梯度截断)
5. **模块五**：Gated RNN - GRU 门控单元手撕 (重置门、更新门与梯度直连高速公路)
6. **模块六**：LSTM 核心单元手撕 (遗忘门、输入门、输出门与细胞状态加法通道)
7. **模块七**：自回归循环神经网络语言模型 (RNNLM) 因果预测与交叉熵损失手撕
8. **模块八**：困惑度 (Perplexity, PPL) 评测与温度采样自回归文本生成

---
## 模块一：单步 Vanilla RNNCell 纯 NumPy 白盒前向与反向传播求导

### 【笔试考点与推导闭式】
1. **前向公式**：
   $$h_t = \tanh(x_t W_x + h_{t-1} W_h + b)$$
2. **微积分求导链式法则**：
   设传入梯度为 $dh_t$，激活中间变量为 $a_t = x_t W_x + h_{t-1} W_h + b$：
   $$da_t = dh_t \odot (1 - h_t^2)$$
   $$dW_x = x_t^T \cdot da_t, \quad dW_h = h_{t-1}^T \cdot da_t, \quad db = \sum da_t, \quad dh_{t-1} = da_t \cdot W_h^T$$

In [ ]:
import numpy as np

class VanillaRNNCell:
    def __init__(self, in_dim, hidden_dim):
        self.Wx = np.random.randn(in_dim, hidden_dim) * 0.01
        self.Wh = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b = np.zeros(hidden_dim)
        
        # 梯度容器
        self.dWx = np.zeros_like(self.Wx)
        self.dWh = np.zeros_like(self.Wh)
        self.db = np.zeros_like(self.b)
        self.cache = None

    def forward(self, x, h_prev):
        """
        x: (B, in_dim)
        h_prev: (B, hidden_dim)
        """
        h_next = np.tanh(np.dot(x, self.Wx) + np.dot(h_prev, self.Wh) + self.b)
        self.cache = (x, h_prev, h_next)
        return h_next

    def backward(self, dh_next):
        x, h_prev, h_next = self.cache
        # tanh 导数: (1 - y^2)
        dtanh = dh_next * (1.0 - h_next ** 2)
        
        self.dWx = np.dot(x.T, dtanh)
        self.dWh = np.dot(h_prev.T, dtanh)
        self.db = np.sum(dtanh, axis=0)
        
        dx = np.dot(dtanh, self.Wx.T)
        dh_prev = np.dot(dtanh, self.Wh.T)
        return dx, dh_prev

# 测试单步 RNNCell 梯度求导
cell = VanillaRNNCell(in_dim=4, hidden_dim=3)
x0 = np.random.randn(2, 4)
h0 = np.zeros((2, 3))
h1 = cell.forward(x0, h0)
dx, dh_prev = cell.backward(np.ones_like(h1))
print("前向隐状态 h1 形状:", h1.shape)
print("反向回传 dh_prev 形状:", dh_prev.shape)
assert h1.shape == (2, 3) and dh_prev.shape == (2, 3)
print(">>> 单步 RNNCell 微积分推导验证通过！")

---
## 模块二：TimeRNN 时序展开与 BPTT 全量逆流梯度手撕

### 【笔试考点与陷阱】
1. **时序展开**：按 $t = 0, 1, \dots, T-1$ 顺序推进前向状态；
2. **时序反向传播 (BPTT)**：从时刻 $T-1$ 逆向倒推至时刻 $0$；
3. **梯度累加陷阱**：当前步回传的隐状态梯度不仅包含上一层（如输出分类器）下发的 $dh_t$，还必须**加上未来时刻逆流回来的 $dh_{\text{next}}$**：
   $$dh_t^{\text{total}} = dh_t + dh_{\text{next}}$$

In [ ]:
class TimeRNN:
    def __init__(self, in_dim, hidden_dim):
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.Wx = np.random.randn(in_dim, hidden_dim) * 0.01
        self.Wh = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b = np.zeros(hidden_dim)
        
        self.dWx = np.zeros_like(self.Wx)
        self.dWh = np.zeros_like(self.Wh)
        self.db = np.zeros_like(self.b)
        self.layers = []
        self.h_prev = None

    def forward(self, xs, h_0=None):
        """
        xs: (B, T, in_dim)
        """
        B, T, D = xs.shape
        if h_0 is None:
            h_0 = np.zeros((B, self.hidden_dim))
            
        self.layers = []
        hs = np.empty((B, T, self.hidden_dim), dtype=np.float32)
        h = h_0
        
        for t in range(T):
            cell = VanillaRNNCell(self.in_dim, self.hidden_dim)
            cell.Wx, cell.Wh, cell.b = self.Wx, self.Wh, self.b
            h = cell.forward(xs[:, t, :], h)
            hs[:, t, :] = h
            self.layers.append(cell)
            
        self.h_prev = h_0
        return hs

    def backward(self, dhs):
        """
        dhs: (B, T, hidden_dim)
        """
        B, T, H = dhs.shape
        dxs = np.empty((B, T, self.in_dim), dtype=np.float32)
        
        self.dWx[...] = 0
        self.dWh[...] = 0
        self.db[...] = 0
        dh = np.zeros((B, H), dtype=np.float32)
        
        # BPTT 逆序反向回流
        for t in reversed(range(T)):
            cell = self.layers[t]
            # 必须加上来自未来时刻的隐状态梯度 dh!
            dx, dh = cell.backward(dhs[:, t, :] + dh)
            dxs[:, t, :] = dx
            self.dWx += cell.dWx
            self.dWh += cell.dWh
            self.db += cell.db
            
        return dxs

time_rnn = TimeRNN(4, 3)
xs_dummy = np.random.randn(2, 5, 4) # B=2, T=5, D=4
hs = time_rnn.forward(xs_dummy)
dxs = time_rnn.backward(np.ones_like(hs))
print("TimeRNN 输出 hs 形状:", hs.shape)
print("TimeRNN 反向梯度 dxs 形状:", dxs.shape)
assert hs.shape == (2, 5, 3) and dxs.shape == (2, 5, 4)
print(">>> BPTT 全时序反向传播累加成功！")

---
## 模块三：梯度裁剪 (Gradient Clipping) 算法手撕 (L2 范数安全制动)

### 【笔试考点与公式】
- **对抗梯度爆炸的终极手术刀**：
  $$\text{total\_norm} = \sqrt{\sum_{i} \|g_i\|^2_2}$$
  $$\text{if } \text{total\_norm} > \text{max\_norm}: \quad g_i \leftarrow g_i \cdot \frac{\text{max\_norm}}{\text{total\_norm}}$$$$

In [ ]:
def clip_grads(grads, max_norm=5.0):
    """
    梯度裁剪纯 NumPy 向量化实现
    grads: 包含若干参数梯度矩阵的列表 [g1, g2, ...]
    """
    total_norm = 0.0
    for grad in grads:
        total_norm += np.sum(grad ** 2)
    total_norm = np.sqrt(total_norm)
    
    rate = max_norm / (total_norm + 1e-6)
    if rate < 1.0:
        for grad in grads:
            grad *= rate
    return total_norm

# 模拟暴涨梯度
big_grads = [np.array([100.0, 200.0]), np.array([300.0, 400.0])]
orig_norm = clip_grads(big_grads, max_norm=5.0)
after_norm = np.sqrt(sum(np.sum(g ** 2) for g in big_grads))
print(f"原始失控梯度范数: {orig_norm:.2f}")
print(f"裁剪后安全梯度范数: {after_norm:.2f}")
assert np.isclose(after_norm, 5.0)
print(">>> 梯度裁剪安全制动测试通过！")

---
## 模块四：Truncated BPTT 截断反向传播手撕 (跨 Batch 隐状态传递)

### 【笔试考点与面试话术】
- **无限长文本破局之道**：前向传播时隐状态 $h$ 永不截断跨 Chunk 持续传递；反向传播时在 Chunk 边界执行 **`detach`**（切断计算图），反向搜索责任只查当前小片段，从而实现 $O(1)$ 显存上限。

In [ ]:
def truncated_bptt_demo():
    """
    演示 Truncated BPTT: 隐状态跨 Chunk 传递，梯度在边界截断
    """
    time_rnn = TimeRNN(in_dim=4, hidden_dim=3)
    # 模拟长文本分为 2 个 Chunk
    chunk1 = np.random.randn(2, 5, 4)
    chunk2 = np.random.randn(2, 5, 4)
    
    # 1. Chunk 1 推进
    hs1 = time_rnn.forward(chunk1, h_0=None)
    last_h = hs1[:, -1, :].copy() # 核心: copy/detach, 切断梯度追溯!
    
    # 2. Chunk 2 承接上一个 Chunk 记忆继续推进
    hs2 = time_rnn.forward(chunk2, h_0=last_h)
    
    print("Chunk 1 末状态传入 Chunk 2 首步，记忆连续不断！")
    print("Chunk 2 初始隐状态与 Chunk 1 严格无缝吻合:", np.allclose(last_h, time_rnn.h_prev))
    assert np.allclose(last_h, time_rnn.h_prev)

truncated_bptt_demo()

---
## 模块五：Gated RNN - GRU 门控单元手撕 (直连梯度高速公路)

### 【笔试考点与公式推导】
1. **重置门 (Reset Gate)**：$r = \sigma(x W_{xr} + h_{t-1} W_{hr} + b_r)$；
2. **更新门 (Update Gate)**：$z = \sigma(x W_{xz} + h_{t-1} W_{hz} + b_z)$；
3. **候选状态**：$\tilde{h} = \tanh(x W_{xh} + (r \odot h_{t-1}) W_{hh} + b_h)$；
4. **状态聚合与高速公路**：
   $$h_t = (1 - z) \odot h_{t-1} + z \odot \tilde{h}$$
   反向传播时，误差通过 $(1 - z)$ 通道直接穿透，无需连乘权重矩阵，彻底终结梯度消失！

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20.0, 20.0)))

class SimpleGRUCell:
    def __init__(self, in_dim, hidden_dim):
        self.in_dim = in_dim
        self.H = hidden_dim
        # 打包合并权重矩阵 [r, z, h] 提升矩阵计算吞吐
        self.Wx = np.random.randn(in_dim, 3 * hidden_dim) * 0.02
        self.Wh = np.random.randn(hidden_dim, 3 * hidden_dim) * 0.02
        self.b = np.zeros(3 * hidden_dim)

    def forward(self, x, h_prev):
        H = self.H
        # 拆分三个门分支
        gates_x = np.dot(x, self.Wx) + self.b
        gates_h = np.dot(h_prev, self.Wh)
        
        r = sigmoid(gates_x[:, :H] + gates_h[:, :H])
        z = sigmoid(gates_x[:, H:2*H] + gates_h[:, H:2*H])
        h_tilde = np.tanh(gates_x[:, 2*H:] + np.dot(r * h_prev, self.Wh[:, 2*H:]))
        
        # 核心加权旁路直连
        h_next = (1.0 - z) * h_prev + z * h_tilde
        return h_next, (r, z, h_tilde)

gru = SimpleGRUCell(4, 3)
h_gru, _ = gru.forward(x0, h0)
print("GRU 输出隐状态 h 形状:", h_gru.shape)
assert h_gru.shape == (2, 3)
print(">>> GRU 门控单元前向验证成功！")

---
## 模块六：LSTM 核心单元手撕 (细胞状态加法通道)

### 【笔试考点与四门机制】
- **遗忘门 $f$、输入门 $i$、输出门 $o$、候选记忆 $\tilde{c}$**：
  $$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$
  $$h_t = o_t \odot \tanh(c_t)$$
- **细胞状态 $c_t$ 的线性无损通道**是 LSTM 解决长距离依赖的核心机制。

In [ ]:
class SimpleLSTMCell:
    def __init__(self, in_dim, hidden_dim):
        self.H = hidden_dim
        self.Wx = np.random.randn(in_dim, 4 * hidden_dim) * 0.02
        self.Wh = np.random.randn(hidden_dim, 4 * hidden_dim) * 0.02
        self.b = np.zeros(4 * hidden_dim)

    def forward(self, x, h_prev, c_prev):
        H = self.H
        gates = np.dot(x, self.Wx) + np.dot(h_prev, self.Wh) + self.b
        
        f = sigmoid(gates[:, :H])
        i = sigmoid(gates[:, H:2*H])
        c_tilde = np.tanh(gates[:, 2*H:3*H])
        o = sigmoid(gates[:, 3*H:])
        
        # 核心加法记忆通道
        c_next = f * c_prev + i * c_tilde
        h_next = o * np.tanh(c_next)
        return h_next, c_next

lstm = SimpleLSTMCell(4, 3)
c0 = np.zeros((2, 3))
h_lstm, c_lstm = lstm.forward(x0, h0, c0)
print("LSTM 输出 h 形状与 c 形状:", h_lstm.shape, c_lstm.shape)
assert h_lstm.shape == (2, 3) and c_lstm.shape == (2, 3)
print(">>> LSTM 细胞记忆通道验证成功！")

---
## 模块七：自回归语言模型 (RNNLM) 因果时序错位预测与交叉熵损失手撕

### 【笔试考点与时序错位】
- **自回归本质**：根据前 $t$ 个词预测第 $t+1$ 个词；
- **时序对齐**：
  - 输入：$x = (w_0, w_1, \dots, w_{T-1})$
  - 目标：$y = (w_1, w_2, \dots, w_T)$
  - 严格右移一个 Token 对齐计算交叉熵损失。

In [ ]:
def cross_entropy_loss(logits, targets, eps=1e-8):
    """
    logits: (B * T, V)
    targets: (B * T,) 整数 ID
    """
    # Softmax 归一化
    exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
    
    # 提取真实类别的预测概率
    N = logits.shape[0]
    correct_log_probs = -np.log(probs[np.arange(N), targets] + eps)
    loss = np.sum(correct_log_probs) / N
    return loss, probs

# 模拟时序错位与自回归预测
corpus = np.array([0, 1, 2, 3, 4, 0]) # 6 个词
inputs = corpus[:-1]   # [0, 1, 2, 3, 4]
targets = corpus[1:]   # [1, 2, 3, 4, 0] 时序右移 1 位

dummy_logits = np.random.randn(5, 5) # 5 步预测 5 个词
loss, _ = cross_entropy_loss(dummy_logits, targets)
print("时序自回归因果交叉熵单步 Loss:", round(loss, 4))
assert loss > 0

---
## 模块八：困惑度 (Perplexity, PPL) 评测与温度自回归采样生成

### 【笔试考点与公式】
1. **困惑度定义**：
   $$\text{PPL} = \exp(\mathcal{L}_{\text{cross\_entropy}})$$
   表示模型在预测下一个词时平均在多少个候选词中感到迷茫；
2. **温度采样 (Temperature Scaling)**：
   $$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$
   - $T \to 0$：退化为 Greedy 贪心搜索，稳定保守；
   - $T > 1$：概率分布扁平化，生成多样性高、富有创造力。

In [ ]:
def sample_with_temperature(logits, temperature=1.0):
    """
    基于温度系数从未归一化 logits 中采样下一个 Token
    """
    scaled_logits = logits / max(temperature, 1e-4)
    exp_s = np.exp(scaled_logits - np.max(scaled_logits))
    probs = exp_s / np.sum(exp_s)
    
    return np.random.choice(len(probs), p=probs)

# 快速验证 PPL 与温度采样
loss_val = 2.3026 # 相当于 -log(0.1)
ppl = np.exp(loss_val)
print(f"交叉熵 Loss={loss_val} 对应的困惑度 PPL={ppl:.2f} (相当于十选一)")

test_logits = np.array([1.0, 2.0, 5.0, 0.5])
greedy_sample = sample_with_temperature(test_logits, temperature=0.01)
creative_sample = sample_with_temperature(test_logits, temperature=2.0)
print("极低温采样 (锁定最大值 ID=2):", greedy_sample)
assert greedy_sample == 2
print("高温度采样结果:", creative_sample)
print(">>> PPL 评测与温度采样自回归验证成功！")

---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. 梯度回传累加: dh_total = dh_layer + dh_future，遗漏未来时刻逆流梯度直接挂科！
2. 梯度裁剪安全阀: total_norm = sqrt(sum(g^2))，超阈值时乘 max_norm/total_norm 等比收缩。
3. 截断 BPTT 智慧: 隐状态 h 跨段无损传递，反向责任边界执行 detach 规避显存爆炸。
4. 门控高速直连: GRU 通过 (1-z)、LSTM 通过 c_t 加法通道直通，终结梯度消失。
5. 因果时序错位: targets = corpus[1:], inputs = corpus[:-1]，PPL = exp(Loss)。
```